In [1]:
import os
import json
from openai import OpenAI
from google.colab import userdata

In [2]:
client = OpenAI(api_key=userdata.get('OPENAI_APIKEY'))

In [3]:
prompt = "Hello , How are you ?"

try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7
        )
        print(response.choices[0].message.content)
except Exception as e:
        print(f"OpenAI Error: {e}")

Hello! I'm just a computer program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?


In [4]:
import json
import http.client


def get_current_weather(location: str) -> str:
    """
    Get current weather in a given location  .
    Args:
    	location: City Name like London, California.
    """
    print(f"-> TOOL ACTIVATED: get_current_weather for {location}...")

    conn = http.client.HTTPSConnection("weather-api167.p.rapidapi.com")

    headers = {
      'x-rapidapi-key': "1a58725e79msh9fbb6b9a7daf4e3p169acejsnae4454a41e6b",
      'x-rapidapi-host': "weather-api167.p.rapidapi.com",
      'Content-Type': "application/json",
      'Accept': "application/json"
    }

    conn.request("GET", "/api/weather/forecast?place="+location+"%2CGB&cnt=3&units=standard&type=three_hour&mode=json&lang=en", headers=headers)

    res = conn.getresponse()
    data = res.read()

    return json.dumps(data.decode("utf-8"))


# Map of available function names to the actual Python functions
AVAILABLE_FUNCTIONS = {
    "get_current_weather": get_current_weather
}

In [5]:
get_current_weather("Manchester")

-> TOOL ACTIVATED: get_current_weather for Manchester...


'"{\\"cod\\":\\"200\\",\\"message\\":0,\\"cnt\\":3,\\"list\\":[{\\"dt\\":1784948400,\\"summery\\":\\"overcast clouds, 287.05K, feels like 286.69K, winds 2.27 meter/sec from West-Southwest, 84% humidity\\",\\"main\\":{\\"temprature\\":287.05,\\"temprature_feels_like\\":286.69,\\"temprature_min\\":287.05,\\"temprature_max\\":287.79,\\"temprature_unit\\":\\"K\\",\\"pressure\\":1012,\\"sea_level_pressure\\":1012,\\"ground_level_pressure\\":1001,\\"pressure_unit\\":\\"hPa\\",\\"humidity\\":84,\\"humidity_unit\\":\\"%\\",\\"temp_kf\\":-0.74},\\"weather\\":[{\\"id\\":804,\\"main\\":\\"Clouds\\",\\"description\\":\\"overcast clouds\\",\\"icon\\":\\"https://openweathermap.org/img/wn/04n@2x.png\\"}],\\"clouds\\":{\\"cloudiness\\":95,\\"unit\\":\\"%\\"},\\"wind\\":{\\"speed\\":2.27,\\"degrees\\":252,\\"direction\\":\\"West-Southwest\\",\\"gust_speed\\":8.48,\\"speed_unit\\":\\"meter/sec\\"},\\"visibility_distance\\":10000,\\"visibility_unit\\":\\"m\\",\\"probability_of_precipitation\\":0,\\"proba

In [6]:
# This is how the AI model learns about the tools it can use.
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get current weather in a given location.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "City Name like London, California"},
                },
                "required": ["location"],
            },
        },
    }
]

In [11]:
import json
import http.client



def run_agent(user_prompt: str):
    """
    The main execution loop for the Getting Weather Report Agent.
    """
    print(f"\n--- Running Getting Weather Report Agent ---")

    system_prompt = (
        "You are an Weather Report Agent"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    collected_data = {}

    while True:
        print("\n[AI Thinking...]")
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=tools_schema,
            tool_choice="auto",
        )

        response_message = response.choices[0].message
        messages.append(response_message)

        if response_message.tool_calls:
            tool_calls = response_message.tool_calls

            for tool_call in tool_calls:
                function_name = tool_call.function.name
                function_to_call = AVAILABLE_FUNCTIONS.get(function_name)
                function_args = json.loads(tool_call.function.arguments)

                if not function_to_call:
                    print(f"Error: Unknown function {function_name}")
                    continue

                # Execute the function
                function_result = function_to_call(**function_args)



                # Append the tool result as a NEW message
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": function_result
                })

        else:
            print("\n--- FINAL AGENT SUMMARY ---")
            print(response_message)
            break

In [12]:
user_prompt = "London"

print("\n" + "="*80 + "\n")
run_agent(f"What is the weather of {user_prompt}")
print("\n" + "="*80 + "\n")




--- Running Getting Weather Report Agent ---

[AI Thinking...]
-> TOOL ACTIVATED: get_current_weather for London...

[AI Thinking...]

--- FINAL AGENT SUMMARY ---
ChatCompletionMessage(content="Here's the current weather in London:\n\n1. **Time:** 03:00\n   - **Weather:** Broken clouds\n   - **Temperature:** 290.39K\n   - **Feels Like:** 290.02K\n   - **Wind:** 2.36 m/s, from North-Northwest\n   - **Humidity:** 71%\n   - **Cloudiness:** 72%\n   - **Part of Day:** Night\n   - ![Weather Icon](https://openweathermap.org/img/wn/04n@2x.png)\n\n2. **Time:** 06:00\n   - **Weather:** Broken clouds\n   - **Temperature:** 290.16K\n   - **Feels Like:** 289.64K\n   - **Wind:** 1.09 m/s, from West-Northwest\n   - **Humidity:** 66%\n   - **Cloudiness:** 60%\n   - **Part of Day:** Day\n   - ![Weather Icon](https://openweathermap.org/img/wn/04d@2x.png)\n\n3. **Time:** 09:00\n   - **Weather:** Scattered clouds\n   - **Temperature:** 294.23K\n   - **Feels Like:** 293.67K\n   - **Wind:** 3.04 m/s, fro